# Treinamento dos modelos

Treina os modelos tradicionais (Linear SVC e Regressão Logística) no texto limpo e ajusta o BERTimbau no texto original. Os dois usam a mesma partição gravada por `main_preprocessamento.ipynb`. Os artefatos são salvos no Google Drive para uso em `main.ipynb`.

## Ambiente de execução

Antes de iniciar, disponibilize `src/` e `requirements.txt` neste ambiente do Colab e selecione um runtime com GPU. O fine-tuning do BERTimbau exige CUDA.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%pip install -q -r requirements.txt evaluate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


## Carregamento da partição

Lê `splits/treino.csv` e `teste.csv`. Os modelos tradicionais recebem `clean_text`. O BERTimbau recebe `texto_completo`.

In [2]:
from pathlib import Path

from src.data import carregar_split

DATASET_DIR = Path('/content/drive/MyDrive/Fake.br-Corpus')
RESULTADOS_DIR = DATASET_DIR / 'resultados'

treino, teste = carregar_split(DATASET_DIR / 'splits')
X_train = treino['clean_text']
X_test = teste['clean_text']
y_train = treino['label']
y_test = teste['label']
X_train_bert = treino['texto_completo']
X_test_bert = teste['texto_completo']


## Conferência da partição

A partição já está gravada, com as mesmas notícias para os dois paradigmas. Esta célula só confirma o tamanho e o balanceamento dos subconjuntos.

In [5]:
print(f'Treino: {len(y_train):,} registros')
print(f'Teste: {len(y_test):,} registros')
print(f'Classe treino: {y_train.value_counts().sort_index().to_dict()}')
print(f'Classe teste: {y_test.value_counts().sort_index().to_dict()}')

Treino: 5,760 registros
Teste: 1,440 registros
Classe treino: {0: 2880, 1: 2880}
Classe teste: {0: 720, 1: 720}


## Modelos tradicionais

Treina TF-IDF, Linear SVC e Regressão Logística, grava os artefatos em `Fake.br-Corpus/modelos/` e o custo de treino em `Fake.br-Corpus/resultados/custo_treino.csv`.

In [1]:
!pip install --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.0/639.0 kB 17.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 97.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 11.0 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 re

In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
from src.train_modelos_tradicionais import treinar_modelos_tradicionais

resultado_classicos = treinar_modelos_tradicionais(
    X_train,
    X_test,
    y_train,
    modelos_dir=DATASET_DIR / 'modelos',
    resultados_dir=RESULTADOS_DIR,
)

print('Modelos tradicionais treinados e persistidos.')
for caminho in resultado_classicos['caminhos'].values():
    print(caminho)
print('Custos de treino:')
for linha in resultado_classicos['custos']:
    print(linha)


Modelos tradicionais treinados e persistidos.
/content/drive/MyDrive/Fake.br-Corpus/modelos/vectorizer_tfidf.joblib
/content/drive/MyDrive/Fake.br-Corpus/modelos/modelo_svc.joblib
/content/drive/MyDrive/Fake.br-Corpus/modelos/modelo_lr.joblib
Custos de treino:
{'Modelo': 'Linear SVC', 'etapa': 'treino', 'n_amostras': 5760, 'tempo_s': 14.0376, 'tempo_cpu_s': 13.9008, 'ram_antes_mb': 333.0625, 'ram_depois_mb': 495.6836, 'ram_delta_mb': 163.6055, 'ram_pico_mb': 698.8125, 'vram_pico_mb': None, 'tempo_por_noticia_ms': 2.4371, 'tamanho_artefato_mb': 0.2219}
{'Modelo': 'Logistic Regression', 'etapa': 'treino', 'n_amostras': 5760, 'tempo_s': 14.0747, 'tempo_cpu_s': 13.9378, 'ram_antes_mb': 333.0625, 'ram_depois_mb': 495.6836, 'ram_delta_mb': 163.0664, 'ram_pico_mb': 698.8125, 'vram_pico_mb': None, 'tempo_por_noticia_ms': 2.4435, 'tamanho_artefato_mb': 0.2221}


## Fine-tuning do BERTimbau

Executa o fine-tuning no texto original, com a GPU selecionada, e salva o melhor modelo em `Fake.br-Corpus/bertimbau_fake_br/modelo_final/`. O custo de treino é gravado em `Fake.br-Corpus/resultados/custo_treino.csv`. Esta etapa pode levar bastante tempo.

In [3]:
import pandas as pd

from src.bert_finetuning import finetune_bertimbau

df = pd.concat([treino, teste], ignore_index=True)
resultado_bert = finetune_bertimbau(
    df=df,
    X_train=X_train_bert,
    X_test=X_test_bert,
    y_train=y_train,
    y_test=y_test,
    output_dir=str(DATASET_DIR / 'bertimbau_fake_br'),
    resultados_dir=RESULTADOS_DIR,
)
print('Custo de treino do BERTimbau:')
for linha in resultado_bert['custos']:
    print(linha)


torch: 2.11.0+cu130
torch.version.cuda: 13.0
cuda built: True
cuda available: True
nvidia-smi: GPU 0: Tesla T4 (UUID: GPU-19939b44-c1f7-0ef9-132c-8ed69c362d33)
GPU ativa: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo com sucesso em: /content/drive/MyDrive/Fake.br-Corpus/bertimbau_fake_br/modelo_final
Custo de treino do BERTimbau:
{'Modelo': 'BERTimbau', 'etapa': 'treino', 'n_amostras': 5760, 'tempo_s': 558.2445, 'tempo_cpu_s': 473.115, 'ram_antes_mb': 1914.6641, 'ram_depois_mb': 2244.6328, 'ram_delta_mb': 329.9688, 'ram_pico_mb': 2630.3555, 'vram_pico_mb': 4744.0767, 'tempo_por_noticia_ms': 96.9175, 'hf_train_runtime': 556.8284, 'tamanho_artefato_mb': 416.19}
